In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data" / "clean"

standings_path = DATA_DIR / "standings.csv"
matches_path = DATA_DIR / "team_matches_clean.csv"
players_path = DATA_DIR / "players_clean.csv"

In [3]:
standings = pd.read_csv(standings_path)
matches = pd.read_csv(matches_path)
players = pd.read_csv(players_path)

In [4]:
print("Standings:", standings.shape)
print("Matches:", matches.shape)
print("Players:", players.shape)

Standings: (20, 10)
Matches: (760, 15)
Players: (551, 56)


In [5]:
teams = sorted(matches["team"].dropna().unique())

dim_team = pd.DataFrame({
    "TeamID": range(1, len(teams) + 1),
    "TeamName": teams
})

dim_team

,TeamID,TeamName
0,1,Arsenal
1,2,Aston Villa
2,3,Bournemouth
3,4,Brentford
4,5,Brighton
5,6,Burnley
6,7,Chelsea
7,8,Crystal Palace
8,9,Everton
9,10,Fulham


In [6]:
assert len(dim_team) == 20
assert dim_team["TeamName"].is_unique

In [7]:
team_mapping = dict(
    zip(dim_team["TeamName"], dim_team["TeamID"])
)

team_mapping

{'Arsenal': 1,
 'Aston Villa': 2,
 'Bournemouth': 3,
 'Brentford': 4,
 'Brighton': 5,
 'Burnley': 6,
 'Chelsea': 7,
 'Crystal Palace': 8,
 'Everton': 9,
 'Fulham': 10,
 'Leeds': 11,
 'Liverpool': 12,
 'Manchester City': 13,
 'Manchester United': 14,
 'Newcastle United': 15,
 'Nottingham Forest': 16,
 'Sunderland': 17,
 'Tottenham': 18,
 'West Ham': 19,
 'Wolverhampton Wanderers': 20}

In [8]:
fact_matches = matches.copy()

In [9]:
fact_matches["TeamID"] = fact_matches["team"].map(team_mapping)

In [10]:
fact_matches[
    ["team", "TeamID"]
].drop_duplicates().sort_values("TeamID")

,team,TeamID
0,Arsenal,1
38,Aston Villa,2
76,Bournemouth,3
114,Brentford,4
152,Brighton,5
190,Burnley,6
228,Chelsea,7
266,Crystal Palace,8
304,Everton,9
342,Fulham,10


In [11]:
assert fact_matches["TeamID"].notna().all()

In [12]:
fact_matches["date"] = pd.to_datetime(
    fact_matches["date"]
)

In [13]:
fact_matches["MatchID"] = (
    fact_matches["date"].dt.strftime("%Y%m%d")
    + "_"
    + fact_matches.apply(
        lambda row: "_".join(
            sorted([row["team"], row["opponent"]])
        ),
        axis=1
    )
)

In [14]:
fact_matches["MatchID"].nunique()

380

In [15]:
assert fact_matches["MatchID"].nunique() == 380

In [16]:
match_counts = fact_matches.groupby("MatchID").size()

match_counts.value_counts()

2    380
Name: count, dtype: int64

In [27]:
# ==========================================
# CREATE DIM DATE
# ==========================================

# Đảm bảo Date có cùng kiểu dữ liệu
fact_matches["Date"] = pd.to_datetime(fact_matches["Date"]).dt.normalize()

# Tạo calendar liên tục
date_range = pd.date_range(
    start=fact_matches["Date"].min(),
    end=fact_matches["Date"].max(),
    freq="D"
)

dim_date = pd.DataFrame({
    "Date": date_range
})

# Các thuộc tính thời gian
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["DayName"] = dim_date["Date"].dt.day_name()

In [28]:
# Kiểm tra mỗi ngày có thuộc đúng 1 matchweek hay không
date_matchweek_check = (
    fact_matches
    .groupby("Date")["matchweek"]
    .nunique()
)

date_matchweek_check.value_counts()

matchweek
1    98
2    16
Name: count, dtype: int64

In [29]:
matchweek_mapping = (
    fact_matches
    .groupby("Date")["matchweek"]
    .first()
)

In [30]:
dim_date["Matchweek"] = (
    dim_date["Date"].map(matchweek_mapping)
)

In [31]:
print("Shape:", dim_date.shape)
print("Unique Date:", dim_date["Date"].is_unique)
print("Duplicate Date:", dim_date["Date"].duplicated().sum())

Shape: (283, 7)
Unique Date: True
Duplicate Date: 0


In [32]:
dim_date.head(10)

,Date,Year,Month,MonthName,Day,DayName,Matchweek
0,2025-08-15,2025,8,August,15,Friday,1.0
1,2025-08-16,2025,8,August,16,Saturday,1.0
2,2025-08-17,2025,8,August,17,Sunday,1.0
3,2025-08-18,2025,8,August,18,Monday,1.0
4,2025-08-19,2025,8,August,19,Tuesday,NaN
5,2025-08-20,2025,8,August,20,Wednesday,NaN
6,2025-08-21,2025,8,August,21,Thursday,NaN
7,2025-08-22,2025,8,August,22,Friday,2.0
8,2025-08-23,2025,8,August,23,Saturday,2.0
9,2025-08-24,2025,8,August,24,Sunday,2.0


In [33]:
dim_date.to_csv(
    DATA_DIR / "dim_date.csv",
    index=False
)

In [34]:
fact_matches[
    [
        "MatchID",
        "Date",
        "team",
        "opponent",
        "venue",
        "result",
        "goals_for",
        "goals_against",
        "xg_for",
        "xg_against"
    ]
].head(10)

,MatchID,Date,team,opponent,venue,result,goals_for,goals_against,xg_for,xg_against
0,20250817_Arsenal_Manchester United,2025-08-17,Arsenal,Manchester United,Away,W,1,0,1.330400,1.376530
1,20250823_Arsenal_Leeds,2025-08-23,Arsenal,Leeds,Home,W,5,0,2.737610,0.156761
2,20250831_Arsenal_Liverpool,2025-08-31,Arsenal,Liverpool,Away,L,0,1,0.485679,0.481404
3,20250913_Arsenal_Nottingham Forest,2025-09-13,Arsenal,Nottingham Forest,Home,W,3,0,2.881700,0.293681
4,20250921_Arsenal_Manchester City,2025-09-21,Arsenal,Manchester City,Home,D,1,1,0.718575,1.122700
5,20250928_Arsenal_Newcastle United,2025-09-28,Arsenal,Newcastle United,Away,W,2,1,2.070480,0.739851
6,20251004_Arsenal_West Ham,2025-10-04,Arsenal,West Ham,Home,W,2,0,3.058500,0.864549
7,20251018_Arsenal_Fulham,2025-10-18,Arsenal,Fulham,Away,W,1,0,2.078230,0.502332
8,20251026_Arsenal_Crystal Palace,2025-10-26,Arsenal,Crystal Palace,Home,W,1,0,0.856175,0.350534
9,20251101_Arsenal_Burnley,2025-11-01,Arsenal,Burnley,Away,W,2,0,2.516490,0.679177


In [35]:
home_matches = fact_matches[
    fact_matches["venue"].str.lower() == "home"
].copy()

away_matches = fact_matches[
    fact_matches["venue"].str.lower() == "away"
].copy()

In [36]:
print("Home:", len(home_matches))
print("Away:", len(away_matches))

Home: 380
Away: 380


In [37]:
fact_match = home_matches[
    [
        "MatchID",
        "Date",
        "matchweek",
        "team",
        "opponent",
        "goals_for",
        "goals_against",
        "xg_for",
        "xg_against",
        "result"
    ]
].copy()

In [38]:
fact_match = fact_match.rename(
    columns={
        "team": "HomeTeam",
        "opponent": "AwayTeam",
        "goals_for": "HomeGoals",
        "goals_against": "AwayGoals",
        "xg_for": "HomeXG",
        "xg_against": "AwayXG",
        "result": "HomeResult"
    }
)

In [39]:
fact_match.shape

(380, 10)

In [40]:
fact_match["MatchID"].nunique()

380

In [41]:
fact_match["MatchResult"] = fact_match["HomeResult"].map({
    "W": "Home Win",
    "D": "Draw",
    "L": "Away Win"
})

In [42]:
fact_match["GoalDifference"] = (
    fact_match["HomeGoals"]
    - fact_match["AwayGoals"]
)

In [44]:
fact_match["xGDifference"] = (
    fact_match["HomeXG"]
    - fact_match["AwayXG"]
)

In [45]:
fact_match["HomeTeamID"] = (
    fact_match["HomeTeam"].map(team_mapping)
)

In [46]:
fact_match["AwayTeamID"] = (
    fact_match["AwayTeam"].map(team_mapping)
)

In [47]:
assert fact_match["HomeTeamID"].notna().all()
assert fact_match["AwayTeamID"].notna().all()

In [48]:
fact_match = fact_match[
    [
        "MatchID",
        "Date",
        "matchweek",
        "HomeTeamID",
        "HomeTeam",
        "AwayTeamID",
        "AwayTeam",
        "HomeGoals",
        "AwayGoals",
        "HomeXG",
        "AwayXG",
        "GoalDifference",
        "xGDifference",
        "HomeResult",
        "MatchResult"
    ]
]

In [49]:
print("Rows:", len(fact_match))
print("Unique matches:", fact_match["MatchID"].nunique())
print("Duplicate MatchID:", fact_match["MatchID"].duplicated().sum())
print("Missing Home Team:", fact_match["HomeTeam"].isna().sum())
print("Missing Away Team:", fact_match["AwayTeam"].isna().sum())

Rows: 380
Unique matches: 380
Duplicate MatchID: 0
Missing Home Team: 0
Missing Away Team: 0


In [50]:
total_home_goals = fact_match["HomeGoals"].sum()
total_away_goals = fact_match["AwayGoals"].sum()

total_goals = total_home_goals + total_away_goals

print("Home goals:", total_home_goals)
print("Away goals:", total_away_goals)
print("Total goals:", total_goals)

Home goals: 580
Away goals: 465
Total goals: 1045


In [51]:
fact_match["MatchResult"].value_counts()

MatchResult
Home Win    162
Away Win    114
Draw        104
Name: count, dtype: int64

In [52]:
assert fact_match["MatchResult"].notna().all()
assert fact_match["MatchResult"].value_counts().sum() == 380

In [53]:
fact_match.to_csv(
    DATA_DIR / "fact_match.csv",
    index=False
)

In [54]:
print("Rows:", len(fact_match))
print("Unique matches:", fact_match["MatchID"].nunique())
print("Duplicate MatchID:", fact_match["MatchID"].duplicated().sum())
print("Total goals:", fact_match["HomeGoals"].sum() + fact_match["AwayGoals"].sum())
print(fact_match["MatchResult"].value_counts())

Rows: 380
Unique matches: 380
Duplicate MatchID: 0
Total goals: 1045
MatchResult
Home Win    162
Away Win    114
Draw        104
Name: count, dtype: int64


In [55]:
fact_players = players.copy()

In [56]:
fact_players.columns.tolist()

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'G+A-PK',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 'PK_stats_shooting',
 'PKatt_stats_shooting',
 'Crs',
 'TklW',
 'Int',
 'Fld',
 'CrdY_stats_misc',
 'CrdR_stats_misc',
 '2CrdY',
 'Fls',
 'OG',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKatt_stats_keeper',
 'PKA',
 'PKsv',
 'PKm',
 'Ast_per90',
 'Sh_per90',
 'Gls_per90']

In [57]:
print("Rows:", len(fact_players))
print("Columns:", len(fact_players.columns))

Rows: 551
Columns: 56


In [58]:
fact_players["Player"].is_unique

False

In [59]:
fact_players.columns.tolist()

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'G+A-PK',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 'PK_stats_shooting',
 'PKatt_stats_shooting',
 'Crs',
 'TklW',
 'Int',
 'Fld',
 'CrdY_stats_misc',
 'CrdR_stats_misc',
 '2CrdY',
 'Fls',
 'OG',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKatt_stats_keeper',
 'PKA',
 'PKsv',
 'PKm',
 'Ast_per90',
 'Sh_per90',
 'Gls_per90']

In [61]:
duplicate_players = (
    fact_players[
        fact_players["Player"].duplicated(keep=False)
    ]
    .sort_values("Player")
)

duplicate_players[
    ["Player", "Nation", "Squad", "Pos", "Age", "MP", "Starts", "Min"]
]

,Player,Nation,Squad,Pos,Age,MP,Starts,Min
500,Adama Traoré,es ESP,Fulham,MF,29.0,15,1,274
499,Adama Traoré,es ESP,West Ham,"FW,MF",29.0,9,0,121
457,Antoine Semenyo,gh GHA,Manchester City,"FW,MF",25.0,17,17,1397
456,Antoine Semenyo,gh GHA,Bournemouth,MF,25.0,20,20,1798
255,Brennan Johnson,wls WAL,Crystal Palace,"FW,MF",24.0,18,13,1112
254,Brennan Johnson,wls WAL,Tottenham,"FW,MF",24.0,16,6,610
310,Douglas Luiz,br BRA,Nottingham Forest,MF,27.0,8,5,315
309,Douglas Luiz,br BRA,Aston Villa,MF,27.0,13,7,604
156,Eberechi Eze,eng ENG,Arsenal,"FW,MF",27.0,32,21,1812
157,Eberechi Eze,eng ENG,Crystal Palace,MF,27.0,1,1,83


In [62]:
print("Unique players:", fact_players["Player"].nunique())
print("Duplicate rows:", fact_players["Player"].duplicated().sum())

Unique players: 537
Duplicate rows: 14


In [63]:
duplicate_players["Player"].value_counts()

Player
Adama Traoré            2
Antoine Semenyo         2
Brennan Johnson         2
Douglas Luiz            2
Eberechi Eze            2
Evann Guessand          2
Facundo Buonanotte      2
Harvey Elliott          2
James Ward-Prowse       2
Jørgen Strand Larsen    2
Marc Guiu               2
Marc Guéhi              2
Oscar Bobb              2
Tyrique George          2
Name: count, dtype: int64

In [64]:
duplicate_players[
    [
        "Player",
        "Nation",
        "Pos",
        "Squad",
        "Age",
        "MP",
        "Starts",
        "Min",
        "Gls",
        "Ast"
    ]
].to_string(index=False)

'              Player  Nation   Pos                   Squad  Age  MP  Starts  Min  Gls  Ast\n        Adama Traoré  es ESP    MF                  Fulham 29.0  15       1  274    0    0\n        Adama Traoré  es ESP FW,MF                West Ham 29.0   9       0  121    0    0\n     Antoine Semenyo  gh GHA FW,MF         Manchester City 25.0  17      17 1397    7    1\n     Antoine Semenyo  gh GHA    MF             Bournemouth 25.0  20      20 1798   10    3\n     Brennan Johnson wls WAL FW,MF          Crystal Palace 24.0  18      13 1112    0    1\n     Brennan Johnson wls WAL FW,MF               Tottenham 24.0  16       6  610    2    0\n        Douglas Luiz  br BRA    MF       Nottingham Forest 27.0   8       5  315    0    0\n        Douglas Luiz  br BRA    MF             Aston Villa 27.0  13       7  604    1    0\n        Eberechi Eze eng ENG FW,MF                 Arsenal 27.0  32      21 1812    7    2\n        Eberechi Eze eng ENG    MF          Crystal Palace 27.0   1       1   8

In [65]:
duplicate_players[
    ["Player", "Squad"]
].sort_values(["Player", "Squad"])

,Player,Squad
500,Adama Traoré,Fulham
499,Adama Traoré,West Ham
456,Antoine Semenyo,Bournemouth
457,Antoine Semenyo,Manchester City
255,Brennan Johnson,Crystal Palace
254,Brennan Johnson,Tottenham
309,Douglas Luiz,Aston Villa
310,Douglas Luiz,Nottingham Forest
156,Eberechi Eze,Arsenal
157,Eberechi Eze,Crystal Palace


In [66]:
fact_players = fact_players.copy()

fact_players["PlayerID"] = (
    fact_players["Player"].astype(str).str.strip()
    + "_"
    + fact_players["Squad"].astype(str).str.strip()
)

In [67]:
print("Rows:", len(fact_players))
print("Unique PlayerID:", fact_players["PlayerID"].nunique())

Rows: 551
Unique PlayerID: 551


In [68]:
assert fact_players["PlayerID"].is_unique

In [69]:
fact_players["TeamID"] = (
    fact_players["Squad"].map(team_mapping)
)

In [70]:
print(
    "Missing TeamID:",
    fact_players["TeamID"].isna().sum()
)

Missing TeamID: 0


In [71]:
transfer_players = (
    fact_players
    .groupby("Player")["Squad"]
    .nunique()
)

transfer_players = transfer_players[
    transfer_players > 1
]

transfer_players

Player
Adama Traoré            2
Antoine Semenyo         2
Brennan Johnson         2
Douglas Luiz            2
Eberechi Eze            2
Evann Guessand          2
Facundo Buonanotte      2
Harvey Elliott          2
James Ward-Prowse       2
Jørgen Strand Larsen    2
Marc Guiu               2
Marc Guéhi              2
Oscar Bobb              2
Tyrique George          2
Name: Squad, dtype: int64

In [72]:
cols = fact_players.columns.tolist()

cols.remove("PlayerID")
cols.remove("TeamID")

fact_players = fact_players[
    ["PlayerID", "TeamID"] + cols
]

In [73]:
fact_players.head()

,PlayerID,TeamID,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,...,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,Ast_per90,Sh_per90,Gls_per90
0,Brenden Aaronson_Leeds,11,1,Brenden Aaronson,us USA,"FW,MF",Leeds,eng Premier League,24.0,2000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.183748,1.727236,0.146999
1,Jerome Abbey_Wolverhampton Wanderers,20,2,Jerome Abbey,eng ENG,MF,Wolverhampton Wanderers,eng Premier League,15.0,2009.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
2,Zach Abbott_Nottingham Forest,16,3,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
3,Tammy Abraham_Aston Villa,2,16,Tammy Abraham,eng ENG,FW,Aston Villa,eng Premier League,27.0,1997.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,2.832168,0.629371
4,Joshua Acheampong_Chelsea,7,22,Joshua Acheampong,eng ENG,DF,Chelsea,eng Premier League,19.0,2006.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.540541,0.135135


In [74]:
fact_players.to_csv(
    DATA_DIR / "fact_players.csv",
    index=False
)

In [75]:
print("Rows:", len(fact_players))
print("Unique PlayerID:", fact_players["PlayerID"].nunique())
print("Missing TeamID:", fact_players["TeamID"].isna().sum())
print("PlayerID unique:", fact_players["PlayerID"].is_unique)

Rows: 551
Unique PlayerID: 551
Missing TeamID: 0
PlayerID unique: True


In [82]:
fact_standings = standings.copy()

In [83]:
print("Rows:", len(fact_standings))
print("Columns:", len(fact_standings.columns))

Rows: 20
Columns: 10


In [78]:
fact_standings.columns.tolist()

['Rank', 'team', 'MP', 'W', 'D', 'L', 'GF', 'GA', 'Pts', 'GD']

In [84]:
fact_standings = fact_standings.rename(
    columns={"team": "TeamName"}
)

In [85]:
fact_standings["TeamID"] = (
    fact_standings["TeamName"].map(team_mapping)
)

In [86]:
print("Rows:", len(fact_standings))
print("Missing TeamID:", fact_standings["TeamID"].isna().sum())
print("Unique TeamID:", fact_standings["TeamID"].nunique())

Rows: 20
Missing TeamID: 0
Unique TeamID: 20


In [87]:
print("Min Rank:", fact_standings["Rank"].min())
print("Max Rank:", fact_standings["Rank"].max())
print("Unique Rank:", fact_standings["Rank"].nunique())

Min Rank: 1
Max Rank: 20
Unique Rank: 20


In [88]:
assert set(fact_standings["Rank"]) == set(range(1, 21))

In [89]:
fact_standings["Pts_Calc"] = (
    fact_standings["W"] * 3
    + fact_standings["D"]
)

In [90]:
(fact_standings["Pts"] == fact_standings["Pts_Calc"]).value_counts()

True    20
Name: count, dtype: int64

In [91]:
fact_standings.drop(
    columns=["Pts_Calc"],
    inplace=True
)

In [92]:
fact_standings["GD_Calc"] = (
    fact_standings["GF"] - fact_standings["GA"]
)

In [93]:
(fact_standings["GD"] == fact_standings["GD_Calc"]).value_counts()

True    20
Name: count, dtype: int64

In [94]:
fact_standings.drop(
    columns=["GD_Calc"],
    inplace=True
)

In [95]:
fact_standings = fact_standings[
    [
        "TeamID",
        "TeamName",
        "Rank",
        "MP",
        "W",
        "D",
        "L",
        "GF",
        "GA",
        "GD",
        "Pts"
    ]
]

In [96]:
print("Rows:", len(fact_standings))
print("Unique TeamID:", fact_standings["TeamID"].nunique())
print("Duplicate TeamID:", fact_standings["TeamID"].duplicated().sum())
print("Missing TeamID:", fact_standings["TeamID"].isna().sum())

Rows: 20
Unique TeamID: 20
Duplicate TeamID: 0
Missing TeamID: 0


In [97]:
fact_standings.to_csv(
    DATA_DIR / "fact_standings.csv",
    index=False
)

In [98]:
(fact_standings["Pts"] == fact_standings["W"] * 3 + fact_standings["D"]).value_counts()

True    20
Name: count, dtype: int64

In [99]:
(fact_standings["GD"] == fact_standings["GF"] - fact_standings["GA"]).value_counts()

True    20
Name: count, dtype: int64

In [100]:
print("=" * 50)
print("FINAL DATA MODEL CHECK")
print("=" * 50)

print(f"DimTeam:        {len(dim_team)} rows")
print(f"DimDate:        {len(dim_date)} rows")
print(f"FactMatch:      {len(fact_match)} rows")
print(f"FactMatches:    {len(fact_matches)} rows")
print(f"FactPlayers:    {len(fact_players)} rows")
print(f"FactStandings:  {len(fact_standings)} rows")

print("=" * 50)

print("DimTeam unique:", dim_team["TeamName"].is_unique)
print("DimDate unique:", dim_date["Date"].is_unique)
print("FactMatch unique:", fact_match["MatchID"].is_unique)
print("FactPlayers unique:", fact_players["PlayerID"].is_unique)
print("FactStandings unique:", fact_standings["TeamID"].is_unique)

FINAL DATA MODEL CHECK
DimTeam:        20 rows
DimDate:        283 rows
FactMatch:      380 rows
FactMatches:    760 rows
FactPlayers:    551 rows
FactStandings:  20 rows
DimTeam unique: True
DimDate unique: True
FactMatch unique: True
FactPlayers unique: True
FactStandings unique: True


In [101]:
dim_team.head()

,TeamID,TeamName
0,1,Arsenal
1,2,Aston Villa
2,3,Bournemouth
3,4,Brentford
4,5,Brighton


In [102]:
dim_team.columns.tolist()

['TeamID', 'TeamName']

In [103]:
dim_team.to_csv(
    DATA_DIR / "dim_team.csv",
    index=False
)

In [104]:
print("Rows:", len(dim_team))
print("Unique TeamID:", dim_team["TeamID"].is_unique)
print("Unique TeamName:", dim_team["TeamName"].is_unique)

Rows: 20
Unique TeamID: True
Unique TeamName: True


In [105]:
print(fact_matches.shape)
print(fact_matches.columns.tolist())

(760, 18)
['date', 'team', 'opponent', 'points', 'goals_for', 'goals_against', 'xg_for', 'xg_against', 'venue', 'result', 'goal_diff', 'xg_diff', 'matchweek', 'cum_points', 'form_pts_last5', 'TeamID', 'MatchID', 'Date']


In [106]:
fact_matches.to_csv(
    DATA_DIR / "fact_matches.csv",
    index=False
)

In [107]:
print("Rows:", len(fact_matches))
print("Unique MatchID:", fact_matches["MatchID"].nunique())
print("Unique TeamID:", fact_matches["TeamID"].nunique())
print("Missing TeamID:", fact_matches["TeamID"].isna().sum())
print("Missing MatchID:", fact_matches["MatchID"].isna().sum())
print("Missing Date:", fact_matches["Date"].isna().sum())

Rows: 760
Unique MatchID: 380
Unique TeamID: 20
Missing TeamID: 0
Missing MatchID: 0
Missing Date: 0
